# Decider 2B on a free GPU
Runtime → Change runtime type → **T4 GPU**. Run all cells, copy the `trycloudflare.com` URL, then on your laptop:

```bash
export STRANDS_DECIDER_URL=https://<printed>.trycloudflare.com
python -m referee.game
```
The tunnel is public and unauthenticated while this notebook runs: stop the runtime when you are done.

In [ ]:
!pip -q install "strands-decider[cuda]"
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
import subprocess, time, urllib.request
subprocess.Popen('strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8099 > serve.log 2>&1', shell=True)
while True:
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8099/health').read().decode()); break
    except Exception:
        time.sleep(5)

In [ ]:
import re
subprocess.Popen('./cloudflared tunnel --url http://127.0.0.1:8099 > tunnel.log 2>&1', shell=True)
url = None
while not url:
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    url = m and m.group(0)
print('export STRANDS_DECIDER_URL=' + url)

In [ ]:
# Sanity check + latency on this GPU
import json
body = json.dumps({'state': 'INVENTORY: rusty dagger. PLAYER: I swing my legendary sword.', 'questions': {'grounded': {'type': 'noul', 'instructions': 'Does the player only use items in INVENTORY?'}}}).encode()
for _ in range(3):
    r = urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8099/v1/systemone', data=body, headers={'content-type': 'application/json'}))
    print(json.loads(r.read()))